# 🚦 Traffic Demand Prediction
*An Elite Ensemble Model (LightGBM + XGBoost) featuring Cyclical Time, Smoothed Target Encoding, and Peak-Demand Weighting.*

In [15]:
%pip install scikit-learn
%pip install lightgbm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [16]:
%pip install pandas
%pip install numpy
%pip install xgboost scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
import os
import multiprocessing
# Prevents joblib crashes on Windows
os.environ['LOKY_MAX_CPU_COUNT'] = str(multiprocessing.cpu_count() or 4)

import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import VotingRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
print("Environment Ready for Final Submission Generation!")

Environment Ready for Final Submission Generation!


In [18]:
def generate_ultimate_submission():
    print("Loading Train and Test datasets...")
    train = pd.read_csv('./dataset/train.csv')
    test = pd.read_csv('./dataset/test.csv')
    
    train_len = len(train)
    test_indices = test['Index'].copy()
    
    print("Applying Winning Feature Engineering...")
    # --- 1. Smoothed Target Encoding (Fitted on Train ONLY) ---
    global_mean = train['demand'].mean()
    agg = train.groupby('geohash')['demand'].agg(['count', 'mean']).reset_index()
    smoothing = 10
    agg['smoothed_demand'] = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
    agg = agg[['geohash', 'smoothed_demand']]
    
    # Combine train and test to apply all other features consistently
    combined = pd.concat([train, test], sort=False).reset_index(drop=True)
    
    # Merge the smoothed demand
    combined = pd.merge(combined, agg, on='geohash', how='left')
    combined['smoothed_demand'] = combined['smoothed_demand'].fillna(global_mean)
    
    # --- 2. Cyclical Time Features & Rush Hour ---
    combined['timestamp'] = pd.to_datetime(combined['timestamp'], format='%H:%M', errors='coerce')
    combined['hour'] = combined['timestamp'].dt.hour.fillna(0).astype(int)
    combined['minute'] = combined['timestamp'].dt.minute.fillna(0).astype(int)
    
    combined['hour_sin'] = np.sin(2 * np.pi * combined['hour'] / 24.0)
    combined['hour_cos'] = np.cos(2 * np.pi * combined['hour'] / 24.0)
    combined['minute_sin'] = np.sin(2 * np.pi * combined['minute'] / 60.0)
    combined['minute_cos'] = np.cos(2 * np.pi * combined['minute'] / 60.0)
    
    combined['is_rush_hour'] = combined['hour'].apply(lambda x: 1 if x in [7, 8, 9, 17, 18, 19] else 0)
    combined['day_of_week'] = combined['day'] % 7
    combined['is_weekend'] = combined['day_of_week'].apply(lambda x: 1 if x in [5, 6] else 0)
    
    # The "Perfect Storm" Feature Cross
    combined['geo_rush_cross'] = combined['geohash'].astype(str) + "_" + combined['is_rush_hour'].astype(str)
    
    combined.drop(['timestamp', 'hour', 'minute'], axis=1, inplace=True)
    
    print("Cleaning missing values...")
    # --- 3. Missing Values ---
    combined['RoadType'] = combined['RoadType'].fillna('Unknown')
    combined['Weather'] = combined['Weather'].fillna('Unknown')
    combined['Temperature'] = combined['Temperature'].fillna(combined['Temperature'].median())
    combined['NumberofLanes'] = combined['NumberofLanes'].fillna(combined['NumberofLanes'].mode()[0])
    combined['LargeVehicles'] = combined['LargeVehicles'].fillna('Unknown')
    combined['Landmarks'] = combined['Landmarks'].fillna('Unknown')
    
    print("Encoding categories...")
    # --- 4. Encoding ---
    cat_cols = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'geo_rush_cross']
    for col in cat_cols:
        combined[col] = combined[col].astype(str)
        le = LabelEncoder()
        combined[col] = le.fit_transform(combined[col])
        
    # Split exactly back into Train and Test
    train_processed = combined.iloc[:train_len].copy()
    test_processed = combined.iloc[train_len:].copy()
    
    X_train = train_processed.drop(['Index', 'demand'], axis=1)
    y_train = train_processed['demand']
    X_test = test_processed.drop(['Index', 'demand'], axis=1)
    
    return X_train, y_train, X_test, test_indices

print("Function Loaded! Ready to process data.")

Function Loaded! Ready to process data.


### Model Training and Validation Workflow
This cell block trains the model on 80% of the training data, evaluates it on a 20% validation split, and then refits the ensemble on the full training dataset before generating final predictions for the test set.


In [19]:
# 1. Process the data
X_train, y_train, X_test, test_indices = generate_ultimate_submission()

# 2. Load the 100-Trial Optuna Parameters
lgb_params = {    
    'n_estimators': 500,    
    'learning_rate': 0.15557591569719165,    
    'num_leaves': 85,    
    'max_depth': 7,    
    'min_child_samples': 13,    
    'subsample': 0.7339498255865238,    
    'colsample_bytree': 0.9772210751245085,    
    'objective': 'regression',    
    'random_state': 42,    
    'verbosity': -1
}

xgb_params = {    
    'n_estimators': 363,    
    'learning_rate': 0.09215538167220788,    
    'max_depth': 10,
    'objective': 'reg:squarederror',
    'random_state': 42,    
    'n_jobs': -1
}

# 3. Create the models
model_lgb = lgb.LGBMRegressor(**lgb_params)
model_xgb = xgb.XGBRegressor(**xgb_params)

# 4. Combine them into the Voting Committee
ensemble_model = VotingRegressor([
    ('lightgbm', model_lgb),
    ('xgboost', model_xgb)
])

# 5. Create a validation split and train on the train fold
print("\nSplitting training data for validation...")
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)
train_weights = np.where(y_tr > 0.8, 3.0, 1.0)
print("Training on 80% of data and validating on 20%...")
ensemble_model.fit(X_tr, y_tr, sample_weight=train_weights)
print("Training Complete!")

# 6. Evaluate on the validation set
val_predictions = ensemble_model.predict(X_val)
print(f"Validation set size: {len(X_val)}")

from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
val_r2 = r2_score(y_val, val_predictions)
val_mae = mean_absolute_error(y_val, val_predictions)
val_mse = mean_squared_error(y_val, val_predictions)
val_rmse = np.sqrt(val_mse)

print("\n--- VALIDATION METRICS ---")
print(f"R-Squared (R2):       {val_r2:.4f}")
print(f"Mean Absolute Error:  {val_mae:.4f}")
print(f"Root Mean Squared Error: {val_rmse:.4f}")
print("--------------------------")

# 7. Refit on full train data before predicting the test set
print("Refitting ensemble on full training data before final submission...")
full_train_weights = np.where(y_train > 0.8, 3.0, 1.0)
ensemble_model.fit(X_train, y_train, sample_weight=full_train_weights)
print("Refit Complete!")


Loading Train and Test datasets...
Applying Winning Feature Engineering...
Cleaning missing values...
Encoding categories...

Splitting training data for validation...
Training on 80% of data and validating on 20%...
Training Complete!
Validation set size: 15460

--- VALIDATION METRICS ---
R-Squared (R2):       0.9536
Mean Absolute Error:  0.0205
Root Mean Squared Error: 0.0306
--------------------------
Refitting ensemble on full training data before final submission...
Refit Complete!


### Final Test Set Prediction and Submission
This cell predicts the actual test dataset after the model has been retrained on 100% of the training data, then writes the submission CSV file.


In [20]:
print("Generating Predictions for test.csv...")
predictions = ensemble_model.predict(X_test)

# Assemble the final submission format
submission = pd.DataFrame({
    'Index': test_indices,
    'demand': np.clip(predictions, a_min=0, a_max=None) # Hard floor at 0
})

# Display the first few rows so you can see your final masterpiece
display(submission.head())

# Save directly to your dataset folder
os.makedirs('./output', exist_ok=True)
output_path = './output/ULTIMATE_submission.csv'
submission.to_csv(output_path, index=False)

print(f"\n🎉 SUCCESS! Your final, highly-optimized predictions are ready.")
print(f"Saved to: {output_path}")

Generating Predictions for test.csv...


,Index,demand
0,0,0.075919
1,1,0.019277
2,2,0.028897
3,3,0.023543
4,4,0.050529



🎉 SUCCESS! Your final, highly-optimized predictions are ready.
Saved to: ./output/ULTIMATE_submission.csv
